In [1]:
from src.utils.notebook_setup import *
import src.utils.notebook_ploting as nb_plot
import src.utils.features as features
setup_pandas()

In [2]:
df = load_dataset()

In [3]:
numerical_features, categorical_features = features.split_features(df)
features_df = df[numerical_features].copy()

In [4]:
corr_matrix = features_df.corr(method="pearson")

In [12]:
nb_plot.correlation_heatmap(corr_matrix, is_abs = True)

In [6]:
# limiar para comparação. threshold => 0.9 -> alta correlação
    # talvez seja necessário buscar justificativa para esse valor
threshold = 0.9
# correlação e correlação negativa tem o mesmo impacto
corr_abs = corr_matrix.abs()

# pega apenas parte de cima da matriz, tambémn ignora a diagonal. evita cálculo repetido
#ex:
# 1 2 3 -> nan  2   3
# 4 5 6 -> nan nan  3
# 7 8 9 -> nan nan nan

# np.ones(corr_abs.shape) -> cria matriz de 1 com o shape
# np.triu( -> pega apenas triângulo superior
# k = 1 exclui diagonal
# transforma em boleano
# em resumo, uma máscara boleana para a matriz

upper = corr_abs.where(
    np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
)

# upper.stack() -> empilha as colunas para transformar em linhas, já filtra nan
# .reset_index() -> transforma series em tabela
# renomear os nomes de coluna que o pandas atribui
#display(upper.stack())
corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={
        "level_0": "feature_1",
        "level_1": "feature_2",
        0: "correlation"
    })
)

high_corr_pairs = corr_pairs[
    corr_pairs["correlation"] >= threshold
].sort_values(by="correlation", ascending=False)


In [7]:
high_corr_pairs


,feature_1,feature_2,correlation
15,bidirectional_packets,bidirectional_bytes,1.00
57,bidirectional_stddev_ps,bidirectional_max_ps,0.99
43,bidirectional_mean_ps,bidirectional_stddev_ps,0.94
85,bidirectional_mean_piat_ms,bidirectional_stddev_piat_ms,0.93
99,bidirectional_stddev_piat_ms,bidirectional_max_piat_ms,0.92
44,bidirectional_mean_ps,bidirectional_max_ps,0.92


In [8]:
from src.configs.paths import ARTIFACTS_DIR
from src.io.io_utils import save_parquet

path = ARTIFACTS_DIR / "high_correlation_pairs.parquet"
save_parquet(high_corr_pairs, path)


Salvo em C:\tcc\network-ids-ml-generalization\artifacts\high_correlation_pairs.parquet
